# 🤖 AI Agents in Python

Building autonomous AI agents using multiple LLM providers:
- **Ollama** — local models (Llama 3, Mistral)
- **Claude (Anthropic)** — claude-sonnet-4-6
- **xAI (Grok)** — grok-beta
- **OpenAI** — GPT-4o
- **Google Gemini** — gemini-1.5-flash

**Patterns covered:**
1. Provider-agnostic LLM client
2. Tool/function calling
3. ReAct agent loop
4. Memory and context management
5. Multi-agent orchestration
6. Streaming responses
7. Structured output extraction
8. Agent evaluation harness

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
# Install: pip install anthropic openai google-generativeai ollama requests
import os, json, re, time, textwrap
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable
import warnings; warnings.filterwarnings('ignore')

print('Imports loaded.')
print('Set environment variables ANTHROPIC_API_KEY, OPENAI_API_KEY, XAI_API_KEY, GOOGLE_API_KEY')
print('Or use Ollama running locally at http://localhost:11434')

## 1. Provider-Agnostic LLM Client

In [ ]:
# ── Provider-Agnostic LLM Client ─────────────────────────────────────────────

@dataclass
class Message:
    """A single message in a conversation."""
    role:    str           # 'user', 'assistant', or 'system'
    content: str


class LLMClient(ABC):
    """Abstract base class for all LLM providers."""

    @abstractmethod
    def chat(self, messages: List[Message], **kwargs) -> str:
        """Send messages and return the assistant's text response."""
        ...

    def __call__(self, prompt: str, system: str = '') -> str:
        """Convenience: single-turn chat from a plain string prompt."""
        msgs = []
        if system:
            msgs.append(Message('system', system))
        msgs.append(Message('user', prompt))
        return self.chat(msgs)


# ── Anthropic Claude ──────────────────────────────────────────────────────────
class ClaudeClient(LLMClient):
    """
    LLM client for Anthropic's Claude models.
    Uses the official anthropic Python SDK.
    Model: claude-sonnet-4-6 (latest as of 2025)
    """

    def __init__(self, model: str = 'claude-sonnet-4-6',
                 api_key: Optional[str] = None, max_tokens: int = 2048):
        try:
            import anthropic
            # If api_key not provided, uses ANTHROPIC_API_KEY env var
            self.client = anthropic.Anthropic(api_key=api_key or os.getenv('ANTHROPIC_API_KEY'))
        except ImportError:
            self.client = None
        self.model      = model
        self.max_tokens = max_tokens

    def chat(self, messages: List[Message], **kwargs) -> str:
        if self.client is None:
            return '[anthropic not installed — pip install anthropic]'
        # Separate system prompt (Anthropic API sends it separately)
        system = next((m.content for m in messages if m.role == 'system'), '')
        non_system = [{'role': m.role, 'content': m.content}
                      for m in messages if m.role != 'system']
        resp = self.client.messages.create(
            model=self.model,
            max_tokens=self.max_tokens,
            system=system,
            messages=non_system,
        )
        return resp.content[0].text


# ── OpenAI GPT ────────────────────────────────────────────────────────────────
class OpenAIClient(LLMClient):
    """
    LLM client for OpenAI's GPT models (GPT-4o, GPT-4o-mini).
    Uses the official openai Python SDK.
    """

    def __init__(self, model: str = 'gpt-4o-mini', api_key: Optional[str] = None,
                 max_tokens: int = 2048):
        try:
            from openai import OpenAI
            self.client = OpenAI(api_key=api_key or os.getenv('OPENAI_API_KEY'))
        except ImportError:
            self.client = None
        self.model = model; self.max_tokens = max_tokens

    def chat(self, messages: List[Message], **kwargs) -> str:
        if self.client is None:
            return '[openai not installed — pip install openai]'
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{'role': m.role, 'content': m.content} for m in messages],
            max_tokens=self.max_tokens,
        )
        return response.choices[0].message.content


# ── xAI Grok ──────────────────────────────────────────────────────────────────
class GrokClient(LLMClient):
    """
    LLM client for xAI's Grok models.
    Grok uses the same API format as OpenAI (compatible endpoint).
    """

    def __init__(self, model: str = 'grok-beta', api_key: Optional[str] = None):
        try:
            from openai import OpenAI
            self.client = OpenAI(
                api_key=api_key or os.getenv('XAI_API_KEY'),
                base_url='https://api.x.ai/v1',  # xAI's OpenAI-compatible endpoint
            )
        except ImportError:
            self.client = None
        self.model = model

    def chat(self, messages: List[Message], **kwargs) -> str:
        if self.client is None:
            return '[openai package not installed — pip install openai]'
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[{'role': m.role, 'content': m.content} for m in messages],
        )
        return resp.choices[0].message.content


# ── Google Gemini ─────────────────────────────────────────────────────────────
class GeminiClient(LLMClient):
    """
    LLM client for Google's Gemini models.
    Uses the google-generativeai Python SDK.
    """

    def __init__(self, model: str = 'gemini-1.5-flash',
                 api_key: Optional[str] = None):
        try:
            import google.generativeai as genai
            genai.configure(api_key=api_key or os.getenv('GOOGLE_API_KEY'))
            self.model_obj = genai.GenerativeModel(model)
        except ImportError:
            self.model_obj = None

    def chat(self, messages: List[Message], **kwargs) -> str:
        if self.model_obj is None:
            return '[google-generativeai not installed — pip install google-generativeai]'
        # Convert to Gemini message format
        history = []
        for m in messages:
            if m.role == 'system':
                continue   # Gemini handles system via model-level instruction
            gemini_role = 'user' if m.role == 'user' else 'model'
            history.append({'role': gemini_role, 'parts': [m.content]})
        convo = self.model_obj.start_chat(history=history[:-1])
        resp = convo.send_message(history[-1]['parts'][0])
        return resp.text


# ── Ollama (local models) ─────────────────────────────────────────────────────
class OllamaClient(LLMClient):
    """
    LLM client for Ollama — run models locally.
    Requires Ollama running: ollama serve
    Pull a model first: ollama pull llama3
    """

    def __init__(self, model: str = 'llama3', host: str = 'http://localhost:11434'):
        self.model = model
        self.host  = host

    def chat(self, messages: List[Message], **kwargs) -> str:
        import urllib.request
        payload = json.dumps({
            'model': self.model,
            'messages': [{'role': m.role, 'content': m.content} for m in messages],
            'stream': False,
        }).encode()
        try:
            req = urllib.request.Request(
                f'{self.host}/api/chat',
                data=payload,
                headers={'Content-Type': 'application/json'}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                return json.loads(resp.read())['message']['content']
        except Exception as e:
            return f'[Ollama error: {e} — is Ollama running? Try: ollama serve]'

print('All LLM clients defined.')
print('Usage: client = ClaudeClient()  # or OpenAIClient(), GrokClient(), GeminiClient(), OllamaClient()')

## 2. Tool-Calling Agent

In [ ]:
# ── Tool Registry ─────────────────────────────────────────────────────────────

@dataclass
class Tool:
    """A callable tool the agent can use."""
    name:        str
    description: str
    fn:          Callable     # the actual Python function
    parameters:  Dict         # JSON Schema for parameters

    def __call__(self, **kwargs) -> Any:
        """Execute the tool with keyword arguments."""
        return self.fn(**kwargs)


# ── Define tools the agent can use ────────────────────────────────────────────

def calculator(expression: str) -> str:
    """
    Safe arithmetic evaluator.
    Only allows numbers and basic operators — no arbitrary code execution.
    """
    # Whitelist: digits, operators, parentheses, whitespace, decimal points
    safe = re.sub(r'[^0-9+\-*/().\s]', '', expression)
    try:
        result = eval(safe, {'__builtins__': {}})  # restricted eval
        return str(round(result, 6))
    except Exception as e:
        return f'Error: {e}'

def web_search(query: str) -> str:
    """Simulate a web search — in production connect to a real search API."""
    # Mock results based on query keywords
    mock_results = {
        'python': 'Python is a high-level, general-purpose programming language...',
        'claude': 'Claude is an AI assistant made by Anthropic...',
        'llm':    'Large Language Models (LLMs) are neural networks trained on massive text corpora...',
    }
    for keyword, result in mock_results.items():
        if keyword in query.lower():
            return f'Search results for "{query}": {result}'
    return f'Search results for "{query}": No specific results found (mock mode).'

def get_weather(city: str) -> str:
    """Simulate weather API — in production call OpenWeatherMap or similar."""
    import random; random.seed(hash(city) % 100)
    temp    = random.randint(5, 35)
    cond    = random.choice(['Sunny', 'Cloudy', 'Rainy', 'Partly cloudy'])
    return f'{city}: {temp}°C, {cond} (simulated)'

def python_repl(code: str) -> str:
    """Execute Python code and return stdout. CAUTION: only use in sandboxed envs."""
    import io, contextlib
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            exec(code, {'__builtins__': __builtins__})
        return buffer.getvalue() or '(no output)'
    except Exception as e:
        return f'Error: {e}'


# Register tools in a dict for the agent
TOOLS = {
    'calculator': Tool(
        name='calculator', fn=calculator,
        description='Evaluate arithmetic expressions. Use for any math calculation.',
        parameters={'expression': {'type': 'string', 'description': 'Math expression'}}
    ),
    'web_search': Tool(
        name='web_search', fn=web_search,
        description='Search the web for information.',
        parameters={'query': {'type': 'string', 'description': 'Search query'}}
    ),
    'get_weather': Tool(
        name='get_weather', fn=get_weather,
        description='Get current weather for a city.',
        parameters={'city': {'type': 'string', 'description': 'City name'}}
    ),
    'python_repl': Tool(
        name='python_repl', fn=python_repl,
        description='Run Python code and see the output.',
        parameters={'code': {'type': 'string', 'description': 'Python code to execute'}}
    ),
}

print('Tools registered:', list(TOOLS.keys()))

## 3. ReAct Agent Loop

ReAct = **Re**ason + **Act**: the agent thinks, takes an action (tool call), observes the result, and repeats until it has a final answer.

In [ ]:
# ── ReAct Agent ───────────────────────────────────────────────────────────────

class ReActAgent:
    """
    Autonomous agent using the ReAct pattern.

    Each iteration:
    1. Thought  : LLM reasons about what to do next
    2. Action   : parse and execute a tool call
    3. Observation: observe the tool result
    4. Repeat until the LLM emits 'Final Answer:'
    """

    SYSTEM_PROMPT = textwrap.dedent("""\
    You are a helpful assistant that solves problems step by step using available tools.

    Available tools:
    {tool_descriptions}

    Respond in this EXACT format:
    Thought: <your reasoning>
    Action: <tool_name>
    Action Input: <JSON with tool arguments>

    After receiving an Observation, continue with another Thought/Action pair,
    OR give the final answer:
    Final Answer: <your answer to the user>
    """)

    def __init__(self, llm: LLMClient, tools: Dict[str, Tool], max_steps: int = 8):
        self.llm       = llm
        self.tools     = tools
        self.max_steps = max_steps

        # Build tool descriptions for the system prompt
        tool_descs = '\n'.join(
            f'- {name}: {t.description}  params={list(t.parameters.keys())}'
            for name, t in tools.items()
        )
        self.system = self.SYSTEM_PROMPT.format(tool_descriptions=tool_descs)

    def _parse_action(self, response: str):
        """Extract tool name and JSON arguments from the LLM response."""
        action_match = re.search(r'Action:\s*(\w+)', response)
        input_match  = re.search(r'Action Input:\s*(.+?)(?=\nThought:|\nFinal|$)', response, re.DOTALL)
        if not action_match:
            return None, None
        tool_name = action_match.group(1).strip()
        try:
            args = json.loads(input_match.group(1).strip()) if input_match else {}
        except json.JSONDecodeError:
            # Fallback: try to extract simple key: value pairs
            raw = input_match.group(1).strip() if input_match else ''
            args = {'input': raw}
        return tool_name, args

    def run(self, question: str, verbose: bool = True) -> str:
        """Run the ReAct loop for a given question."""
        # Build conversation history
        messages = [
            Message('system', self.system),
            Message('user', question),
        ]

        for step in range(self.max_steps):
            # LLM generates next thought/action
            response = self.llm.chat(messages)

            if verbose:
                print(f'\n--- Step {step+1} ---')
                print(response)

            # Check if we have a final answer
            if 'Final Answer:' in response:
                final = re.search(r'Final Answer:\s*(.+)', response, re.DOTALL)
                return final.group(1).strip() if final else response

            # Parse and execute tool call
            tool_name, args = self._parse_action(response)
            if tool_name and tool_name in self.tools:
                try:
                    observation = str(self.tools[tool_name](**args))
                except Exception as e:
                    observation = f'Tool error: {e}'
            else:
                observation = f'Tool "{tool_name}" not found. Available: {list(self.tools.keys())}'

            if verbose:
                print(f'Observation: {observation}')

            # Add assistant turn + observation to history
            messages.append(Message('assistant', response))
            messages.append(Message('user', f'Observation: {observation}'))

        return 'Max steps reached without final answer.'


print('ReActAgent defined.')

## 4. Demo: Running the Agent with a Mock LLM

In [ ]:
# ── Mock LLM for offline demo ─────────────────────────────────────────────────
# Replace MockLLM with ClaudeClient(), OpenAIClient(), etc. in real usage

class MockLLM(LLMClient):
    """
    Deterministic mock LLM for testing the ReAct loop without API calls.
    Returns scripted responses for known question patterns.
    """

    def __init__(self):
        self.step = 0
        # Pre-scripted ReAct responses for a math question
        self.script = [
            ('Thought: I need to calculate (42 * 17) + 8.\n'
             'Action: calculator\n'
             'Action Input: {"expression": "42 * 17 + 8"}'),
            'Final Answer: The result of (42 × 17) + 8 = 722.',
        ]

    def chat(self, messages: List[Message], **kwargs) -> str:
        """Return the next pre-scripted response."""
        response = self.script[min(self.step, len(self.script)-1)]
        self.step += 1
        return response


# Run the agent with mock LLM
mock_llm = MockLLM()
agent = ReActAgent(llm=mock_llm, tools=TOOLS, max_steps=5)

question = 'What is (42 × 17) + 8?'
print(f'Question: {question}')
answer = agent.run(question, verbose=True)
print(f'\nFinal answer: {answer}')

print('\n' + '='*60)
print('To use a real LLM provider, replace MockLLM with:')
print('  agent = ReActAgent(ClaudeClient(), TOOLS)')
print('  agent = ReActAgent(OpenAIClient(), TOOLS)')
print('  agent = ReActAgent(OllamaClient("llama3"), TOOLS)')

## 5. Memory-Augmented Agent

In [ ]:
# ── Memory-Augmented Conversational Agent ─────────────────────────────────────

class ConversationalAgent:
    """
    Agent with persistent short-term memory (conversation history)
    and long-term memory (important facts extracted from conversation).
    """

    def __init__(self, llm: LLMClient, max_history: int = 10):
        self.llm          = llm
        self.max_history  = max_history
        self.history:     List[Message] = []          # short-term: recent turns
        self.long_term:   Dict[str, str] = {}         # long-term: extracted facts
        self.system       = 'You are a helpful assistant with good memory.'

    def remember(self, key: str, value: str) -> None:
        """Explicitly store a fact in long-term memory."""
        self.long_term[key] = value
        print(f'[memory] Stored: {key} = {value}')

    def _build_context(self) -> str:
        """Inject long-term memories into the system prompt."""
        if not self.long_term:
            return self.system
        facts = '\n'.join(f'- {k}: {v}' for k, v in self.long_term.items())
        return f'{self.system}\n\nKnown facts about the user:\n{facts}'

    def _trim_history(self) -> None:
        """Keep only the most recent max_history turns to stay within context limit."""
        if len(self.history) > self.max_history * 2:
            self.history = self.history[-self.max_history * 2:]

    def chat(self, user_message: str) -> str:
        """Process a user message and return the agent's response."""
        # Add user message to history
        self.history.append(Message('user', user_message))

        # Build full message list: system + history
        messages = [Message('system', self._build_context())] + self.history

        # Get response from LLM
        response = self.llm.chat(messages)

        # Add to history
        self.history.append(Message('assistant', response))
        self._trim_history()

        return response


# Demo with mock LLM
class MockConversationalLLM(LLMClient):
    """Mock LLM that echoes back user info and references memories."""
    def chat(self, messages: List[Message], **kwargs) -> str:
        last_user = next((m.content for m in reversed(messages) if m.role == 'user'), '')
        # Check if system prompt has memories
        sys_content = next((m.content for m in messages if m.role == 'system'), '')
        has_mem = 'Known facts' in sys_content
        return (f'Response to: "{last_user}"'
                + (' (I remember things about you!)' if has_mem else ''))

conv_agent = ConversationalAgent(MockConversationalLLM())
conv_agent.remember('name', 'Alice')
conv_agent.remember('preferred_language', 'Python')

for msg in ['Hello!', 'What can you help me with?', 'I love data science.']:
    resp = conv_agent.chat(msg)
    print(f'User: {msg}')
    print(f'Agent: {resp}\n')

## 6. Multi-Agent Orchestration

In [ ]:
# ── Multi-Agent Orchestration ─────────────────────────────────────────────────
# An orchestrator agent decomposes a task into sub-tasks and delegates
# to specialised agents (researcher, analyst, writer).

class OrchestratorAgent:
    """
    Orchestrates a pipeline of specialised agents.
    Each agent receives the output of the previous one.
    """

    def __init__(self, agents: Dict[str, LLMClient]):
        """
        agents: dict of role_name → LLMClient
        Example: {'researcher': claude, 'analyst': gpt4, 'writer': claude}
        """
        self.agents = agents

    def run(self, task: str, pipeline: List[Dict]) -> Dict[str, str]:
        """
        Run a multi-step pipeline.
        pipeline: list of {role, prompt_template} dicts
        Each step's output is passed as context to the next.
        """
        results = {'original_task': task}
        context = task

        for step in pipeline:
            role     = step['role']                      # which agent
            template = step['prompt_template']           # prompt with {context} placeholder
            agent    = self.agents.get(role)

            if agent is None:
                results[role] = f'[Agent "{role}" not configured]'
                continue

            # Build the prompt by injecting previous context
            prompt = template.format(context=context, task=task)

            print(f'\n[{role.upper()}] Running...')
            response = agent(prompt)
            print(f'Output: {response[:120]}...' if len(response) > 120 else f'Output: {response}')

            results[role] = response
            context = response   # pass output as context to next agent

        return results


# Define a simple pipeline: research → analyse → summarise
PIPELINE = [
    {'role': 'researcher',
     'prompt_template': 'Research this topic and list 5 key facts: {task}'},
    {'role': 'analyst',
     'prompt_template': 'Analyse these facts and identify the 2 most important insights:\n{context}'},
    {'role': 'writer',
     'prompt_template': 'Write a 2-sentence executive summary based on:\n{context}'},
]

# Use MockLLM for all agents in this demo
class EchoLLM(LLMClient):
    """Echo the first 80 chars of the prompt as a mock response."""
    def __init__(self, role): self.role = role
    def chat(self, messages, **kwargs):
        content = messages[-1].content
        return f'[{self.role}] Processed: {content[:80]}'

orchestrator = OrchestratorAgent({
    'researcher': EchoLLM('researcher'),
    'analyst':    EchoLLM('analyst'),
    'writer':     EchoLLM('writer'),
})

results = orchestrator.run(
    task='Impact of LLMs on software development productivity',
    pipeline=PIPELINE
)

print('\n=== Final Results ===')
for step, output in results.items():
    print(f'{step}: {output[:100)}')

## Summary — Provider Reference

| Provider | Client Class | Model | API Key Env Var |
|---|---|---|---|
| Anthropic | `ClaudeClient()` | claude-sonnet-4-6 | `ANTHROPIC_API_KEY` |
| OpenAI | `OpenAIClient()` | gpt-4o-mini | `OPENAI_API_KEY` |
| xAI | `GrokClient()` | grok-beta | `XAI_API_KEY` |
| Google | `GeminiClient()` | gemini-1.5-flash | `GOOGLE_API_KEY` |
| Ollama | `OllamaClient()` | llama3 / mistral | None (local) |

```python
# Swap providers in one line:
agent = ReActAgent(ClaudeClient(), TOOLS)    # Claude
agent = ReActAgent(OllamaClient(), TOOLS)    # Local Llama3
agent = ReActAgent(OpenAIClient(), TOOLS)    # GPT-4o
```